In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [2]:
# import os
# import numpy as np
# import tensorflow as tf
# import keras
# from keras import layers, models
# from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from keras.utils import image_dataset_from_directory

### Config

In [3]:
DATASET_DIR  = "./dataset_split"          
MODEL_DIR    = "./model"
MODEL_PATH   = os.path.join(MODEL_DIR, "pneumonia_densenet.keras")  
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 25
LEARNING_RATE = 1e-4

os.makedirs(MODEL_DIR, exist_ok=True)

### Data Generators

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,       # Slight rotation is okay
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,   # STRICTLY FALSE for X-Rays
    fill_mode="nearest",
)

val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",   
    shuffle=True,
)

val_gen = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

test_gen = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

Found 4099 images belonging to 2 classes.
Found 877 images belonging to 2 classes.
Found 880 images belonging to 2 classes.


### Calculating class weights for imbalance

In [5]:
normal_count = 1341
pneumonia_count = 3875
total_images = normal_count + pneumonia_count

weight_for_0 = (1 / normal_count) * (total_images / 2.0)
weight_for_1 = (1 / pneumonia_count) * (total_images / 2.0)
class_weights = {0: weight_for_0, 1: weight_for_1}

print(f"Applying Class Weights: Normal: {weight_for_0:.2f}, Pneumonia: {weight_for_1:.2f}")

Applying Class Weights: Normal: 1.94, Pneumonia: 0.67


### Model Training using Transfer learning with DenseNet121

In [6]:
def build_model():
    base_model = tf.keras.applications.DenseNet121(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,      
        weights="imagenet",     
    )
    base_model.trainable = False  

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),  
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            "accuracy", 
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"), # Added
            tf.keras.metrics.Recall(name="recall")        # Added
        ],
    )

    return model, base_model

model, base_model = build_model()

### Callbacks

In [7]:
callbacks = [
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_auc", # Monitoring AUC is better than accuracy for imbalance
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

### Training with Frozen base

In [8]:
print("\n[PHASE 1] Training classifier head...")
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    class_weight=class_weights, # Applying the penalty weights here
    callbacks=callbacks,
)


[PHASE 1] Training classifier head...


ImportError: This requires the scipy module. You can install it via `pip install scipy`